# The Internet Access Map Is Upside Down When You Adjust For Population

*A step-by-step Python tutorial using OWID internet and population data to build two choropleth maps that tell opposite stories*

**Article:** (link added when the article is published)

Run the cells in order. Dataset files are in the same folder as this notebook.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected+png"  # interactive in Jupyter, image on GitHub

### The Dataset

We're combining two datasets from Our World in Data: the ITU internet penetration series ([HERE](https://ourworldindata.org/grapher/share-of-individuals-using-the-internet)) and UN population estimates ([HERE](https://ourworldindata.org/grapher/population)). Both download as CSVs directly from the OWID chart pages.

In [ ]:
import pandas as pd

internet = pd.read_csv("data/internet_pct.csv")
internet.columns = ["entity", "code", "year", "pct_online"]

pop = pd.read_csv("data/population.csv")
pop.columns = ["entity", "code", "year", "population"]

# Filter to 2024 country-level data
inet = internet[
    (internet["code"].str.len() == 3) &
    (internet["year"] == 2024)
].copy()

pop_2023 = pop[
    (pop["code"].str.len() == 3) &
    (pop["year"] == 2023)
].copy()

merged = inet.merge(pop_2023[["code", "population"]], on="code", how="left")
merged = merged.dropna(subset=["pct_online", "population"])
merged = merged[merged["population"] > 100_000]

merged["pct_offline"] = 100.0 - merged["pct_online"]
merged["unconnected_m"] = merged["population"] * (merged["pct_offline"] / 100) / 1_000_000

### Chart 1: The Standard View

This is the map you see in every digital divide report.

In [ ]:
import plotly.express as px

fig = px.choropleth(
    merged,
    locations="code",
    color="pct_online",
    color_continuous_scale="Blues",
    range_color=[0, 100],
    hover_name="entity",
    labels={"pct_online": "% online"},
)
fig.update_layout(
    title=dict(text="Map 1: Share of population online — 2024", font=dict(size=38)),
    geo=dict(projection_type="natural earth", showocean=True, oceancolor="#EEF4FF"),
    coloraxis_colorbar=dict(title=dict(text="% online"), ticksuffix="%"),
)
fig.write_image("chart_1.png", width=1600, height=950, scale=2)

### Chart 2: The Flip

Same countries. Different column. Watch what happens.

In [ ]:
fig2 = px.choropleth(
    merged,
    locations="code",
    color="unconnected_m",
    color_continuous_scale="Oranges",
    hover_name="entity",
    labels={"unconnected_m": "Unconnected (M)"},
)
fig2.update_layout(
    title=dict(text="Map 2: Absolute number of people without internet — 2024", font=dict(size=38)),
    geo=dict(projection_type="natural earth", showocean=True, oceancolor="#EEF4FF"),
    coloraxis_colorbar=dict(title=dict(text="Unconnected\n(millions)")),
)
fig2.write_image("chart_2.png", width=1600, height=950, scale=2)

### Chart 3: Why Both Maps Are Correct

The scatter plot shows the structural reason these two maps can tell opposite stories.

In [ ]:
import plotly.graph_objects as go

fig3 = go.Figure()

# All countries in grey
fig3.add_trace(go.Scatter(
    x=merged["pct_online"],
    y=merged["unconnected_m"],
    mode="markers",
    marker=dict(color="#B8B8B8", size=8, opacity=0.6),
    text=merged["entity"],
))

# Highlight key outliers
for iso, name, color in [
    ("IND", "India", "#D62728"), ("CHN", "China", "#0072B2"),
    ("NGA", "Nigeria", "#E69F00"), ("PAK", "Pakistan", "#009E73"),
    ("BGD", "Bangladesh", "#56B4E9"),
]:
    row = merged[merged["code"] == iso].iloc[0]
    fig3.add_trace(go.Scatter(
        x=[row["pct_online"]], y=[row["unconnected_m"]],
        mode="markers+text", text=[name],
        marker=dict(color=color, size=14),
        textfont=dict(size=26, color=color),
        textposition="top center",
    ))

fig3.update_layout(
    title=dict(text="High connectivity share ≠ low absolute disconnection", font=dict(size=38)),
    xaxis=dict(title="Share of population online (%)", ticksuffix="%"),
    yaxis=dict(title="People without internet (millions)", type="log"),
    showlegend=False,
)
fig3.write_image("chart_3.png", width=1600, height=1000, scale=2)

### One Function Builds Both Maps

Here's the more useful discovery: both choropleths are structurally identical.

In [ ]:
import folium

def internet_choropleth(data, col, color_scheme, legend_name, filename, bins=None):
    geo_url = "https://raw.githubusercontent.com/python-visualization/folium/master/examples/data/world-countries.json"
    m = folium.Map(location=[20, 0], zoom_start=2, tiles="OpenStreetMap")
    kwargs = dict(
        geo_data=geo_url,
        data=data,
        columns=["code", col],
        key_on="feature.id",
        fill_color=color_scheme,
        fill_opacity=0.75,
        line_opacity=0.2,
        legend_name=legend_name,
        nan_fill_color="lightgrey",
    )
    if bins:
        kwargs["threshold_scale"] = bins
    folium.Choropleth(**kwargs).add_to(m)
    m.save(filename)

# Map 1: % connected — 7 equal bins 0–100
internet_choropleth(merged, "pct_online", "Blues",
    "% of population online (2024)", "internet_pct.html",
    bins=[0, 15, 30, 45, 60, 75, 90, 100])

# Map 2: absolute unconnected — log-ish bins
internet_choropleth(merged, "unconnected_m", "Oranges",
    "People without internet (millions, 2024)", "internet_abs.html",
    bins=[0, 1, 10, 50, 100, 250, 510])